# FutureX-Past — forecasting questions whose outcomes are known

[FutureX](https://arxiv.org/abs/2508.11987) is a live benchmark of real-world forecasting: every
week it asks agents about events that have not happened yet. **FutureX-Past** is its archive of
questions that have since resolved, from January to November 2026: sports results, rankings,
prices, releases. Each question is the dataset's own prompt, word for word, and asks for a
boxed answer (`\boxed{}`) at the end of the reply.

Two Benchmarks read this archive:

- **`futurex_past`** (2,036 questions) is scored by our re-implementation of the FutureX
  authors' current rules, with our house Judge in place of theirs. A score is on their scale
  but is not comparable with their leaderboard. Levels 1-2 (pick one option, pick every
  right option) are marked by rule. Levels 3-4 (a
  number, a name or a ranked list) go to our house Judge, which first decides whether a list is
  a ranking or an unordered set and then compares meanings across wording and language. Numbers
  score by distance: within 5% of the truth earns partial credit. The headline weights the four
  level means 10/20/30/40, as the paper does.
- **`futurex_past_study`** (1,118 questions, the 2026-08-02 release) is scored the way the
  crowd-wisdom study scored it, with the study's own code and no Judge. Answers must equal the
  key as text after case-folding. It exists to replicate that study's numbers, so it sits below
  `futurex_past` on levels 3-4 by design.

**Read a score with this in mind.** The events have already happened, and no search is given.
A model trained after an event may simply know it, so this is a closed-book exam on resolved
events, not the live FutureX leaderboard.

## 0. Before running

The next cell picks where this notebook runs: `TARGET = "local"` (the default)
uses your `screamingface up` stack, and `TARGET = "hosted"` uses
`fusion.screamingface.ai`.

**Local.** Install the local services with
`pip install "screamingface[runtime,notebook]"`, then from a terminal:

```bash
screamingface prepare futurex_past        # first run only: download the questions
screamingface prepare futurex_past_study  # the study's release, for section 4
screamingface up                          # start Gateway :9105, Scoreboard :9106, and Engine :9108
screamingface status
```

Use `screamingface logs` to inspect startup failures and `screamingface down` when
finished. Stack management stays outside the notebook so **Run All** never starts or
stops local services. Working from a checkout? `just local-stack-notebooks` in
`packages/screamingface/` does every local step — assets, stack, and Jupyter — in one
command.

**Hosted.** Nothing to start: `pip install "screamingface[notebook]"` is enough, and
`sf.connect()` opens the login in your browser.

In [ ]:
import screamingface as sf

TARGET = "local"  # "local": your `screamingface up` stack · "hosted": fusion.screamingface.ai

if TARGET == "local":
    sf.configure(engine_url="http://127.0.0.1:9108", scoreboard_url="http://127.0.0.1:9106")
elif TARGET == "hosted":
    sf.configure(
        engine_url="https://fusion.screamingface.ai",
        scoreboard_url="https://leaderboard.screamingface.ai",
    )
else:
    raise ValueError(f'TARGET must be "local" or "hosted", not {TARGET!r}')

sf.connect()

## 1. Read the Benchmark's card

`futurex_past` was written by a contributor outside the team (origin `external`). The card
carries the pinned dataset release, the Judge it pins, and the scoring caveats.

In [ ]:
sf.benchmarks.get("futurex_past")

## 2. Run a few questions with one model

The prompt asks for nothing but the boxed answer, so a short reply is enough; a reasoning model
still needs room to think before it writes the box.

In [ ]:
# No `temperature` here on purpose: several current reasoning models reject the parameter
# outright (see 13_contracteval for the provider-by-provider detail).
PARAMS = {"max_tokens": 4096}

solo = sf.Model(model="openrouter/openai/gpt-5.5", params=PARAMS)
report = sf.evaluate(solo, benchmark="futurex_past", limit=5)
report

## 3. Read the per-question outcomes

Each case carries its level and, on levels 3-4, the route the Judge chose (`legacy_type_a` for
option labels, `legacy_type_b` for a number, a name or a ranking, `unordered_set` for a set). A
reply with no box scores 0; a case whose Judge never gave a readable reply shows as failed.

In [ ]:
for case in report.candidates.only.cases:
    grade = case.grade
    if grade is None or case.failures:
        print(case.case_id, case.status, [failure.code for failure in case.failures])
        continue
    evidence = grade.checks[0].evidence[0]
    level, route = evidence.metadata.get("level"), evidence.metadata.get("route")
    print(case.case_id, grade.score, level, route)

## 4. Rerun the crowd-wisdom study's 100 questions

The study scored its committees on 100 FutureX-Past questions; its best, the three cluster
medoids fused by vote, scored 0.302. `STUDY_CASES` are those questions' positions in
`futurex_past_study`. A run on chosen cases cannot be submitted to the leaderboard. A reply with
no usable answer (no box, a hedge, every option) leaves the score, as it did for a model in the
study, and the run metric `excluded` counts those. The study counted a committee's abstain as 0
instead, so that committee's stored answers score 0.306 here, not 0.302.

In [ ]:
# fmt: off
STUDY_CASES = [
    172, 182, 188, 189, 202, 212, 234, 242, 261, 266, 273, 276, 284, 289, 291, 293, 297,
    311, 338, 341, 365, 378, 386, 397, 401, 403, 427, 431, 438, 439, 454, 468, 494, 501,
    507, 515, 518, 527, 543, 560, 578, 600, 603, 609, 615, 622, 630, 663, 664, 707, 713,
    731, 738, 749, 759, 762, 763, 764, 783, 785, 789, 794, 799, 810, 812, 814, 826, 834,
    836, 839, 847, 852, 878, 887, 896, 906, 909, 923, 957, 959, 971, 976, 979, 989, 991,
    993, 1006, 1013, 1021, 1035, 1040, 1045, 1056, 1058, 1065, 1068, 1070, 1075, 1107,
    1114
]
# fmt: on

# The study called every model at temperature 0. OpenRouter rejects that parameter for
# `openai/gpt-5.5`, so it is left out here; add "temperature": 0.0 for a model that accepts it.
study_model = sf.Model(model="openrouter/openai/gpt-5.5", params={"max_tokens": 8192})
study_report = sf.evaluate(study_model, benchmark="futurex_past_study", cases=STUDY_CASES)
study_report

## 5. Before you scale up

A `limit=N` run is a smoke test, not a ranking: the four levels weigh differently, and five
cases will not sample them evenly. One pass over all 2,036 questions is about 2,036 Candidate
calls per model, plus up to three short Judge calls for each of the 664 level-3/4 questions (route,
then match or rank). The study Benchmark calls no Judge. There is no spend cap in this stack, so
`limit` is the only brake.